In [1]:
import numpy as np
from pymoo.indicators.hv import HV
from bikeroot import BikeNetwork, CostApplicator, Cost, NgaMobrp_Cost, alg_utils, eval_utils
import pickle
import os
from bikeroot import Result, biobjective_label_setting_solve, biobjective_label_correcting_solve, heuristic_enabled_mc_label_setting_solve, ngamobrp_solve, nga3mobrp_solve, moedbrp_solve
from bikeroot import HeuristicCheckDom, HeuristicSkip, HeuristicPruning
import time


locations = ['Spokane, WA', "San Francisco, CA", "Prague", "Aveiro"]
path_distances = [500, 1000, 1500, 3000, 5000, 10000, 15000] #by meters

DOWNLOAD_MAPS = False
GENERATE_CANDIDATES = False
RUN_TEST_1 = False
TEST_REF_POINTS = True

In [2]:

if DOWNLOAD_MAPS:
    completed = pickle.load(open('completed_locations.pkl', 'rb')) if 'completed_locations.pkl' in os.listdir() else []
    for location in locations:
        if location in completed:
            print(f"Skipping {location}, already processed.")
            continue
        print(f"Processing {location}...")
        bike_net = BikeNetwork(location)
        size = max(path_distances) + 5000
        loc_name = location.split(",")[0].lower().replace(" ", "_")
        bike_net.download_data(dist_meters=size, elevation=True, save_file=f'data/{loc_name}_{size}.graphml')
        cost = NgaMobrp_Cost()
        g_with_cost = CostApplicator.apply_cost_functions(bike_net,cost)
        bike_net.update_graph(g_with_cost)
        bike_net.save_graph(f'data/{loc_name}_{size}_withcost.graphml')
        completed.append(location)
        pickle.dump(completed, open('completed_locations.pkl', 'wb'))
    

In [3]:
if GENERATE_CANDIDATES:
    for location in locations:
        print(f"Generating candidates for {location}...")
        loc_name = location.split(",")[0].lower().replace(" ", "_")
        bike_net = BikeNetwork(location)
        bike_net.load_graph(f'data/{loc_name}_20000_withcost.graphml')

        candidates = pickle.load(open(f'{loc_name}_candidates.pkl', 'rb')) if f'{loc_name}_candidates.pkl' in os.listdir() else {}
        if candidates == {}:
            for path_distance in path_distances:
                candidates[path_distance] = {'cost': None, 'start': None, 'end': None}

        all_candidates = pickle.load(open(f'{loc_name}_allcandidates.pkl', 'rb')) if f'{loc_name}_allcandidates.pkl'in os.listdir() else []
        
        while not all([candidates[dist]['cost'] is not None for dist in path_distances]):
            nodes = bike_net.random_nodes(2)
            src, dst = str(nodes[0]), str(nodes[1])
            # print(start, end)
            if src == dst:
                continue
            path = alg_utils.a_star(bike_net.graph, start=src, goal=dst, weight='travel distance')
            if path is None:
                continue
            path_cost = alg_utils.total_path_cost(bike_net.graph, path, 'travel distance')
            # print(path_cost)
            all_candidates.append((src, dst, path_cost))
            for path_distance in path_distances:
                if abs(path_cost - path_distance) < 150:
                    if candidates[path_distance]['cost'] is None or abs(path_cost - path_distance) < abs(candidates[path_distance]['cost'] - path_distance):
                        candidates[path_distance]['cost'] = path_cost
                        candidates[path_distance]['start'] = src
                        candidates[path_distance]['end'] = dst
                        print(f"Updated candidate for {path_distance}m: {src} to {dst} with cost {path_cost}")
                        pickle.dump(candidates, open(f'{loc_name}_candidates.pkl', 'wb'))
            
            if len(all_candidates) % 100 == 0:
                pickle.dump(all_candidates, open(f'{loc_name}_allcandidates.pkl', 'wb'))

    for loc in locations:
        loc_name = loc.split(",")[0].lower().replace(" ", "_")
        candidates = pickle.load(open(f'{loc_name}_candidates.pkl', 'rb'))
        all_candidates = pickle.load(open(f'{loc_name}_allcandidates.pkl', 'rb'))
        print(f"Location: {loc}")
        print(f"Candidates: {candidates}")
        print(f"All Candidates: {len(all_candidates)} paths found")

In [4]:
if RUN_TEST_1:
    location = "Spokane, WA"    
    loc_name = location.split(",")[0].lower().replace(" ", "_")
    candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
    print(candidates)

    bike_net = BikeNetwork(location)
    bike_net.load_graph(f'data/testing_maps/{loc_name}_20000_withcost.graphml')
    distance = 500
    src = str(candidates[distance]['start'])
    dst = str(candidates[distance]['end'])
    criteria = ['travel distance','travel time', 'safety', 'comfort']

    bilabelsetting = {'func': biobjective_label_setting_solve, 'params':{}, 'name': 'biobjective_label_setting_solve'}

    bilabelcorrecting = {'func': biobjective_label_correcting_solve, 'params':{}, 'name': 'biobjective_label_correcting_solve'}

    heuristiclabelset = {
        'func':heuristic_enabled_mc_label_setting_solve,
        'params':{
            'heuristic_skip': HeuristicSkip(type='ellipse',G=bike_net.graph, src=src, dst=dst, ab=3.5),
            'heuristic_prune': None,#HeuristicPruning(type='ratio', dst=dst, G=bike_net.graph, criteria='travel distance', alpha=2),
            'heuristic_checkdom': HeuristicCheckDom(type='epsilon', G=bike_net.graph, criteria=criteria, epsilon=0.00)
            },
        'name': 'heuristic_enabled_mc_label_setting_solve'
        }

    ngamobrpset = {'func':ngamobrp_solve, 'params':{'debug':False}, 'name': 'ngamobrp_solve'}
    nga3mobrpset = {'func':nga3mobrp_solve, 'params':{'debug':False}, 'name': 'nga3mobrp_solve'}
    moedbrpset = {'func':moedbrp_solve, 'params':{'debug':False}, 'name': 'moedbrp_solve'}


    algs = [heuristiclabelset]
    results = pickle.load(open('data/testing/results.pkl', 'rb')) if os.path.isfile('data/testing/results.pkl') else []
    print(f"Loaded {len(results)} previous results.")
    print(f"Running tests for {location} from {src} to {dst} with distance {distance}m and criteria {criteria}")
    for alg in algs:
        print(f"Running algorithm: {alg['name']}")
        start_time = time.time()
        path_out = alg['func'](G=bike_net.graph, src=src, dst=dst, criteria=criteria, **alg['params'])
        end_time = time.time()
        result = Result(
            graph_name = f'data/testing_maps/{loc_name}_20000_withcost.graphml',
            algorithm=alg['name'],
            location=location,
            distance=distance,
            src=src,
            dst=dst,
            criteria=criteria,
            params=alg['params']
        )
        print(path_out, f"Time taken: {end_time - start_time:.2f} seconds")
        result.add_result(path_out,None,end_time - start_time)
        results.append(result)
        pickle.dump(results, open('data/testing/results.pkl', 'wb'))


In [5]:
# results = pickle.load(open('data/testing/results.pkl', 'rb'))
# for result in results:
#     print(result)
#     print(result.times)

In [ ]:
if TEST_REF_POINTS:
    location = "Spokane, WA"    
    loc_name = location.split(",")[0].lower().replace(" ", "_")
    candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
    print(candidates)

    bike_net = BikeNetwork(location)
    bike_net.load_graph(f'data/testing_maps/{loc_name}_20000_withcost.graphml')
    criteria = ['travel distance','travel time', 'safety', 'comfort']
    results = pickle.load(open('data/testing/results.pkl', 'rb')) if os.path.isfile('data/testing/results.pkl') else []
    distance = 500
    src = str(candidates[distance]['start'])
    dst = str(candidates[distance]['end'])
    ref_points = eval_utils.calculate_refpoint(bike_net.graph, src, dst, criteria)
    print(f"Reference points for {src} to {dst}: {ref_points}")

{500: {'cost': 495.9196253304533, 'start': '12145206682', 'end': '12145206629'}, 1000: {'cost': 1119.9724710971027, 'start': '10264014984', 'end': '11698463213'}, 1500: {'cost': 1408.3211673993424, 'start': '51032709', 'end': '8141408064'}, 3000: {'cost': 2867.1092999295615, 'start': '8109814002', 'end': '11053828930'}, 5000: {'cost': 5081.723511479114, 'start': '6928027683', 'end': '9771126816'}, 10000: {'cost': 9924.30493589504, 'start': '50950987', 'end': '8570405487'}, 15000: {'cost': 14961.965701478599, 'start': '8571719055', 'end': '8119239337'}}
criteria_costs: {'travel distance': [495.9196253304533, 497.2650056190604, 535.5074576195136, 516.5473627131748], 'travel time': [108.07088940176479, 105.53349590384228, 99.96609100192808, 100.65066742290611], 'safety': [939.1709869585076, 800.374661785338, 569.5266733687284, 594.0769094320432], 'comfort': [1050.1836465016497, 932.057238318138, 615.8881735730112, 594.0769094320432]}
Reference points for 12145206682 to 12145206629: {'trav

In [15]:
for loc in locations:
    loc_name = loc.split(",")[0].lower().replace(" ", "_")
    candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
    candidates_renamed = {}
    changed =  False
    for dist, data in candidates.items():
        if data['cost'] is not None and 'src' not in data.keys():
            changed = True
            candidates_renamed[dist] = {
                'cost': data['cost'],
                'src': data['start'],
                'dst': data['end']
            }
    if changed:
        pickle.dump(candidates_renamed, open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'wb'))


In [16]:
for loc in locations:
    loc_name = loc.split(",")[0].lower().replace(" ", "_")
    candidates = pickle.load(open(f'data/testing_candidates/{loc_name}_candidates.pkl', 'rb'))
    print(loc_name, candidates)

spokane {500: {'cost': 495.9196253304533, 'src': '12145206682', 'dst': '12145206629'}, 1000: {'cost': 1119.9724710971027, 'src': '10264014984', 'dst': '11698463213'}, 1500: {'cost': 1408.3211673993424, 'src': '51032709', 'dst': '8141408064'}, 3000: {'cost': 2867.1092999295615, 'src': '8109814002', 'dst': '11053828930'}, 5000: {'cost': 5081.723511479114, 'src': '6928027683', 'dst': '9771126816'}, 10000: {'cost': 9924.30493589504, 'src': '50950987', 'dst': '8570405487'}, 15000: {'cost': 14961.965701478599, 'src': '8571719055', 'dst': '8119239337'}}
san_francisco {500: {'cost': 524.3536288969926, 'src': '10092146298', 'dst': '10091704178'}, 1000: {'cost': 1001.7781867161574, 'src': '65290322', 'dst': '2656326359'}, 1500: {'cost': 1558.883636640192, 'src': '4014184980', 'dst': '65312712'}, 3000: {'cost': 2977.353492234677, 'src': '53096571', 'dst': '53038452'}, 5000: {'cost': 4985.712724901512, 'src': '6348551505', 'dst': '4941850530'}, 10000: {'cost': 9998.897464909815, 'src': '1294476985